In [ ]:
import sys
import os

from transformers import AutoTokenizer
import datasets
from functools import partial

# Add the project root directory to the Python path
project_root = os.path.abspath(os.path.join(os.getcwd(), '../..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.utils.dataset_tokenization import process_data

In [2]:
model_path = "MathBite/self_corrective_llama_3.1_8B_final"
tokenizer = AutoTokenizer.from_pretrained(model_path)
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
SPECIAL_INSTRUCTION = """\nNote on Self-Correction: As you generate your response, you may encounter an automated instruction. This indicates a potential error was detected.
- If you see the instruction `[rewrite sentence]`, it means the preceding sentence is incorrect. You must immediately provide a new, corrected version of that sentence.
- If you see the instruction `[rewrite response]`, it means the entire preceding response is incorrect. You must immediately provide a new, complete response from the beginning."""

INSERTION_MARKER = "<|eot_id|><|start_header_id|>user<|end_header_id|>"
DELETION_MARKERS = ["<DEL_S>", "<DEL_A>"]
DELETION_TOKEN_IDS = tokenizer.convert_tokens_to_ids(DELETION_MARKERS)
print(DELETION_TOKEN_IDS)

[128256, 128257]


In [21]:
data_path = "../../dataset/final_train_dataset_s2.json"
dataset = datasets.load_dataset("json", data_files=data_path)

In [22]:
dataset

DatasetDict({
    train: Dataset({
        features: ['input', 'incorrect_response', 'errors', 'hallucinated_text', 'correct_response', 'additional_info'],
        num_rows: 1339
    })
})

In [23]:
del_s_replacement_phrase = "[rewrite sentence]"
del_a_replacement_phrase = "[rewrite response]"

del_s_replacement = tokenizer.encode(del_s_replacement_phrase, add_special_tokens=False)
del_a_replacement = tokenizer.encode(del_a_replacement_phrase, add_special_tokens=False)

print(del_s_replacement)
print(del_a_replacement)

[58, 53573, 11914, 60]
[58, 53573, 2077, 60]


In [ ]:
for i in range(100):
    sample = dataset["train"][i]

    res = process_data(
        sample, 
        tokenizer, 
        SPECIAL_INSTRUCTION, 
        INSERTION_MARKER, 
        DELETION_TOKEN_IDS[0], 
        DELETION_TOKEN_IDS[1],
        del_s_replacement,
        del_a_replacement,
        mask_labels=False)

    hall_text_idx = [i for i, label in enumerate(res["hallucination_labels"]) if label == 1 or label == 2]
    hall_text = [res["input_ids"][i] for i in hall_text_idx]
    token_labels = [res["labels"][i] for i in range(len(res["labels"])) if res["labels"][i] != -100]

    print(str(i) + " Full response:\n" + sample["correct_response"])
    print("================================================")
    print("Hallucinated text:")
    for error in sample["hallucinated_text"]:
        print(error)
    print("================================================")
    print("Hallucinated text from tokenized response:")
    print(tokenizer.decode(hall_text))
    print("================================================")
    print("Tokens model will learn from:")
    print(tokenizer.decode(token_labels))
    print("--------------------------------\n\n")


0 Full response:
Let's denote the initial number of roses in the vase as x. However, the problem doesn't provide the initial number of roses. Therefore, we cannot determine the exact number of roses in the vase after Jessica added 8 more.

Since the initial number of roses is unknown, the problem is unsolvable.
Hallucinated text:
Hallucinated text from tokenized response:

Tokens model will learn from:
Let's denote the initial number of roses in the vase as x. However, the problem doesn't provide the initial number of roses. Therefore, we cannot determine the exact number of roses in the vase after Jessica added 8 more.

Since the initial number of roses is unknown, the problem is unsolvable.<|eot_id|>
--------------------------------


1 Full response:
Let's analyze the problem:

Jenna wants to read 600 pages in 30 days, but she won't read on 4 weekdays (13th to 16th) and 1 day (23rd).<DEL_S> Jenna wants to read 600 pages in 30 days, but she won't read on 4 weekdays (13th to 16th). Sh

In [ ]:
SPECIAL_INSTRUCTION = "\nAs you write your answer, you can correct yourself using these tools: Use <DEL_S> to remove the entire sentence before this token, and <DEL_A> to scrap everything you've written and start again."
SPECIAL_INSTRUCTION = "\nIf you realize you have made a mistake, you must use one of the following tools to correct it: Use <DEL_S> to retract the entire sentence immediately preceding this token. Use <DEL_A> to retract your entire response and start over."

SPECIAL_INSTRUCTION = """\nNote on Self-Correction: As you generate your response, you may encounter an automated instruction. This indicates a potential error was detected.
- If you see the instruction `[rewrite sentence]`, it means the preceding sentence is incorrect. You must immediately provide a new, corrected version of that sentence.
- If you see the instruction `[rewrite response]`, it means the entire preceding response is incorrect. You must immediately provide a new, complete response from the beginning."""

INSERTION_MARKER = "<|eot_id|><|start_header_id|>user<|end_header_id|>"
DELETION_MARKERS = ["<DEL_S>", "<DEL_A>"]
DELETION_TOKEN_IDS = tokenizer.convert_tokens_to_ids(DELETION_MARKERS)

mapper = partial(
    process_data,
    tokenizer=tokenizer,
    special_instruction=SPECIAL_INSTRUCTION,
    insertion_marker=INSERTION_MARKER,
    del_s_token_id=DELETION_TOKEN_IDS[0],
    del_a_token_id=DELETION_TOKEN_IDS[1],
    del_s_replacement=del_s_replacement,
    del_a_replacement=del_a_replacement,
    mask_labels=False,
)

In [26]:
dataset.cleanup_cache_files()

{'train': 3}

In [27]:
tokenized_dataset = dataset.map(mapper, batched=False, load_from_cache_file=False)
tokenized_dataset = tokenized_dataset["train"]
columns_to_remove = [
    "input", "correct_response", "incorrect_response", 
    "additional_info", "errors", "hallucinated_text"
]

tokenized_dataset = tokenized_dataset.remove_columns(columns_to_remove)


Map:   0%|          | 0/1339 [00:00<?, ? examples/s]

In [28]:
tokenized_dataset

Dataset({
    features: ['input_ids', 'attention_mask', 'labels', 'hallucination_labels'],
    num_rows: 1339
})

In [ ]:
split_dataset = tokenized_dataset.train_test_split(test_size=0.1, seed=42)
print(split_dataset)

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels', 'hallucination_labels'],
        num_rows: 1205
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels', 'hallucination_labels'],
        num_rows: 134
    })
})


In [30]:
output_dir = "../../dataset/s2"
split_dataset.save_to_disk(output_dir)

Saving the dataset (0/1 shards):   0%|          | 0/1205 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/134 [00:00<?, ? examples/s]

In [4]:
dataset = datasets.load_from_disk("../../dataset/ts")
train_dataset = dataset["train"]
eval_dataset = dataset["test"]

print(train_dataset)
print(eval_dataset)

Dataset({
    features: ['input_ids', 'attention_mask', 'labels', 'hallucination_labels'],
    num_rows: 1205
})
Dataset({
    features: ['input_ids', 'attention_mask', 'labels', 'hallucination_labels'],
    num_rows: 134
})


In [5]:
del_s_counter = 0
del_a_counter = 0

for i in range(len(train_dataset)):
    sample = train_dataset[i]
    for label in sample["hallucination_labels"]:
        if label == 1:
            del_s_counter += 1
            break
        elif label == 2:
            del_a_counter += 1
            break

print(del_s_counter, del_a_counter)
    

255 196


In [6]:
del_s_counter = 0
del_a_counter = 0

for i in range(len(eval_dataset)):
    sample = eval_dataset[i]
    for label in sample["hallucination_labels"]:
        if label == 1:
            del_s_counter += 1
            break
        elif label == 2:
            del_a_counter += 1
            break

print(del_s_counter, del_a_counter)
    

23 25


In [7]:
del_tokens = ["<DEL_S>", "<DEL_A>"]


for i in range(100):
    print(i)
    sample = train_dataset[i]
    print(tokenizer.decode(sample["input_ids"]), "\n")
    tmp_hall_labels = [e for e in sample["hallucination_labels"] if e != -100]

    for j in range(1, 3):
        hall_text_idx = [i for i, label in enumerate(sample["hallucination_labels"]) if label == j]
        hall_text = [sample["input_ids"][i] for i in hall_text_idx]

        print("--------------------------------\n")
        if hall_text_idx:
            print(f"Deletion token: {del_tokens[j-1]}")
            print(tokenizer.decode(hall_text))
        else:
            print(tmp_hall_labels)
    
    # labels = [sample["labels"][i] for i in range(len(sample["labels"])) if sample["labels"][i] != -100]
    print(sample["labels"][-len(tmp_hall_labels):])
    label = [label for label in sample["labels"] if label != -100]
    print(f"Tokens the model will learn from:\n{tokenizer.decode(label)}")
        
    
    print("########################################################\n\n")

0
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are a specialized question-answering AI. Your task is to give a concise answer to the question using *only* the provided context. Make sure to always give an answer.
Note on Self-Correction: As you generate your response, you may encounter an automated instruction. This indicates a potential error was detected.
- If you see the instruction `[rewrite sentence]`, it means the preceding sentence is incorrect. You must immediately provide a new, corrected version of that sentence.
- If you see the instruction `[rewrite response]`, it means the entire preceding response is incorrect. You must immediately provide a new, complete response from the beginning.<|eot_id|><|start_header_id|>user<|end_header_id|>

Context:
'''
The College of Engineering was established in 1920, however, early courses in civil and mechanical engineering were a part of the College of Science since the 1870s. Today the college, housed in the Fitzpatric

In [40]:
train_dataset[5]

{'input_ids': [128000,
  128006,
  9125,
  128007,
  271,
  2675,
  527,
  264,
  96278,
  15592,
  21651,
  1122,
  13,
  4718,
  3465,
  374,
  311,
  11886,
  279,
  2768,
  7033,
  3575,
  382,
  12763,
  1521,
  7504,
  15884,
  512,
  16,
  13,
  3146,
  2127,
  56956,
  279,
  3575,
  68063,
  5629,
  11,
  3619,
  279,
  2728,
  2038,
  323,
  1148,
  374,
  1694,
  4691,
  627,
  17,
  13,
  3146,
  5733,
  434,
  2092,
  85,
  2968,
  68063,
  31001,
  422,
  279,
  3575,
  374,
  2092,
  24694,
  13,
  362,
  3575,
  2643,
  387,
  7120,
  89197,
  422,
  433,
  596,
  3900,
  31356,
  11,
  5727,
  81523,
  11,
  477,
  37856,
  5995,
  2038,
  627,
  18,
  13,
  3146,
  50,
  4035,
  477,
  83017,
  25,
  1035,
  256,
  482,
  3146,
  2746,
  2092,
  24694,
  68063,
  40665,
  264,
  3094,
  14656,
  30308,
  6425,
  11,
  9204,
  682,
  701,
  33811,
  323,
  29217,
  11,
  323,
  1243,
  9539,
  1614,
  279,
  1620,
  35876,
  4320,
  627,
  256,
  482,
  3146,
  2746,
 